# Lesson 06: Agentic RAG — Iterative Maker-Checker Retrieval

In this notebook, you will:
1. Build a `ChromaDB` knowledge base for travel & market intelligence.
2. Wire an iterative Maker-Checker Agentic RAG loop with `google-genai` (`gemini-2.5-flash`) that self-corrects low-relevance vector queries and malformed SQLite NL2SQL queries.

In [ ]:
!pip install -q -U google-genai chromadb python-dotenv

In [ ]:
import os, sqlite3, chromadb
from google import genai

chroma = chromadb.Client()
col = chroma.get_or_create_collection("travel_rag")
col.upsert(
    ids=["doc1", "doc2"],
    documents=[
        "Contoso TravelPass includes free cancellation up to 24 hours before departure.",
        "Competitor SkyPass charges a $45 cancellation fee on economy fares."
    ]
)

def search_kb(query: str) -> str:
    res = col.query(query_texts=[query], n_results=1)
    return res["documents"][0][0]

initial = search_kb("launch")
print("Turn 1 retrieval:", initial)

# The second turn reformulates an under-specified query.
refined = search_kb("TravelPass cancellation policy 24 hours")
print("Turn 2 retrieval:", refined)

# A separate SQLite tool demonstrates a malformed NL2SQL retry.
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE metrics (product TEXT, q4_revenue INT)")
conn.execute("INSERT INTO metrics VALUES ('TravelPass', 485000)")
try:
    conn.execute("SELECT rev FROM metrics").fetchall()
except sqlite3.OperationalError as err:
    print("SQL Turn 1 rejected:", err)
rows = conn.execute("SELECT q4_revenue FROM metrics").fetchall()
print("SQL Turn 2 result:", rows)